# Custom Processing: `_process`, Co-outputs, and `Figure`

When neither `Reprojector` nor `Rasterizer` fits your use case you override
`_process` directly. This notebook covers:

- Writing a custom `_process` method
- The **co-output pattern** — a single `_process` run that writes multiple
  cached artifacts simultaneously by yielding sibling loaders
- `Figure` — a sibling base class to `Data` for plot outputs that cache to
  `path_figures` instead of `path_cache`

In [ ]:
import os, sys
os.chdir('../../..')
sys.path.insert(0, 'docs/loaders')

In [ ]:
from pathlib import Path

from pygeodata import SpatialSpec, get_config, load, process

get_config().update(
    path_cache=Path('data/processed'),
    path_figures=Path('data/figures'),
)

spec = SpatialSpec.from_raster_file('data/wtd.tif')
print('Target spec:', spec)

## 1. Custom `_process`

Override `_process(self, spec)` for any logic that a processor cannot express:
multi-step computation, loading upstream results, writing derived files.

`_process` must write its output to `self.get_processed_path(spec)` and return
`None` (or nothing) when it produces only a single output.

```python
# docs/loaders/pipeline.py

@dataclass
class LandWaterTableDepth(Data):
    wtd: WaterTableDepthLoader
    mask: CountryMaskLoader

    ext = 'tif'
    driver = RioXArrayDriver()

    def _process(self, spec: SpatialSpec) -> None:
        wtd_da  = load(self.wtd,  spec)
        mask_da = load(self.mask, spec)
        wtd_da.where(mask_da == 1).rio.to_raster(self.get_processed_path(spec))
```

In [ ]:
from pipeline import LandWaterTableDepth, WaterTableDepthLoader, CountryMaskLoader

land_wtd = LandWaterTableDepth(
    wtd=WaterTableDepthLoader(),
    mask=CountryMaskLoader(),
)

da = load(land_wtd, spec)
print(da)
print('NaN fraction (sea + nodata):', float(da.isnull().mean()))

## 2. The co-output pattern

Sometimes a single expensive computation naturally produces several related
outputs — for example, computing the mean and standard deviation of a dataset
in one pass.

Instead of running `_process` twice, **yield sibling loaders** from `_process`.
`pygeodata` will write the state hash and params file for every yielded artifact
in a single run, so all siblings share the same cache entry.

```python
# docs/loaders/pipeline.py

@dataclass
class MeanStdLoader(Data):
    wtd: WaterTableDepthLoader
    mask: CountryMaskLoader
    stat: str   # 'mean' or 'std'

    ext = 'tif'
    driver = RioXArrayDriver()

    def _process(self, spec: SpatialSpec):
        da = load(self.wtd, spec).where(load(self.mask, spec) == 1)

        mean_loader = MeanStdLoader(wtd=self.wtd, mask=self.mask, stat='mean')
        std_loader  = MeanStdLoader(wtd=self.wtd, mask=self.mask, stat='std')

        da.mean(...).rio.to_raster(mean_loader.get_processed_path(spec))
        da.std(...).rio.to_raster(std_loader.get_processed_path(spec))

        yield mean_loader   # pygeodata writes hash + params for both
        yield std_loader
```

Calling `process()` on *either* sibling triggers the run and caches both.
The registry browser shows sibling entries together in the "Co-outputs" card.

In [ ]:
from pipeline import MeanStdLoader

wtd   = WaterTableDepthLoader()
mask  = CountryMaskLoader()

mean_loader = MeanStdLoader(wtd=wtd, mask=mask, stat='mean')
std_loader  = MeanStdLoader(wtd=wtd, mask=mask, stat='std')

# Processing mean triggers _process once, which also writes std
process(mean_loader, spec)

print('mean processed:', mean_loader.is_processed(spec))
print('std  processed:', std_loader.is_processed(spec))   # also True

In [ ]:
da_mean = load(mean_loader, spec)
da_std  = load(std_loader,  spec)
print('Mean WTD (land):', float(da_mean))
print('Std  WTD (land):', float(da_std))

## 3. `Figure`: caching plot outputs

`Figure` is a sibling base class to `Data`. Use it for any output that is a
visualisation rather than a data product:

| | `Data` | `Figure` |
|---|---|---|
| Cache root | `path_cache` | `path_figures` |
| Default ext | (must set) | `'png'` |
| Flat layout option | — | `flatten_figures` |
| `load()` | returns data | not typically called |

Everything else — `_process`, hashing, dependency graphs, the registry browser
— works identically.

```python
# docs/loaders/pipeline.py

class WTDFigure(Figure):
    """Cached matplotlib map of water-table depth."""

    ext = 'png'

    def _process(self, spec: SpatialSpec) -> None:
        da = load(WaterTableDepthLoader(), spec).where(
            load(CountryMaskLoader(), spec) == 1
        )
        fig, ax = plt.subplots(figsize=(8, 5))
        da.plot(ax=ax, cmap='Blues_r')
        fig.savefig(self.get_processed_path(spec), dpi=120)
        plt.close(fig)
```

In [ ]:
from pipeline import WTDFigure

fig_loader = WTDFigure()
print('Figure output path:', fig_loader.get_processed_path(spec))

process(fig_loader, spec)
print('Is processed:', fig_loader.is_processed(spec))

In [ ]:
from IPython.display import Image
Image(str(fig_loader.get_processed_path(spec)))

### `flatten_figures`

By default, figures follow the same nested spec/param directory structure as
data outputs. Setting `flatten_figures=True` places all figures directly in
`path_figures` (or a single subdirectory per class), with the spec and params
encoded in the filename. This makes it easier to browse figures in a file
manager without navigating deep trees.

In [ ]:
from pygeodata import set_config

default_path = fig_loader.get_processed_path(spec)

with set_config(flatten_figures=True):
    flat_path = fig_loader.get_processed_path(spec)

print('Default path (nested):')
print(' ', default_path)
print()
print('Flat path:')
print(' ', flat_path)